# Исследование NPS

# Описание исследования

Заказчик этого исследования — большая телекоммуникационная компания, которая оказывает услуги на территории всего СНГ. Для этого необходимо подготовить дашборд с его итогами.

# Цель исследования

Необходимо определить текущий уровень потребительской лояльности, или NPS (от англ. Net Promoter Score), среди клиентов из России.

# Задачи исследования

1. Подключиться к базе данных исследования.
2. Выгрузить данные исследования.
3. Создать дашборд в Tableau по результатам исследования.
4. Ответить на вопросы с помощью дашборда.

# Исходные данные

Таблица user содержит основную информацию о клиентах:

    user_id	- идентификатор клиента, первичный ключ таблицы
    lt_day - количество дней «жизни» клиента
    age - возраст клиента в годах
    gender_segment - пол клиента (1 – женщина, 0 – мужчина)
    os_name - тип операционной системы
    cpe_type_name - тип устройства
    location_id	- идентификатор домашнего региона клиента, внешний ключ, отсылающий к таблице location
    age_gr_id - идентификатор возрастного сегмента клиента, внешний ключ, отсылающий к таблице age_segment
    tr_gr_id - идентификатор сегмента клиента по объёму потребляемого трафика в месяц, внешний ключ, отсылающий к таблице traffic_segment
    lt_gr_id - идентификатор сегмента клиента по количеству месяцев «жизни», внешний ключ, отсылающий к таблице lifetime_segment
    nps_score - оценка клиента в NPS-опросе (от 1 до 10)

Таблица location содержит справочник территорий, в которых телеком-компания оказывает услуги:

    location_id - идентификатор записи, первичный ключ
    country - страна
    city - город

Таблица age_segment содержит данные о возрастных сегментах клиентов:

    age_gr_id - идентификатор сегмента, первичный ключ
    bucket_min - минимальная граница сегмента
    bucket_max - максимальная граница сегмента
    title - название сегмента

Таблица traffic_segment содержит данные о выделяемых сегментах по объёму потребляемого трафика:

    tr_gr_id - идентификатор сегмента, первичный ключ
    bucket_min - минимальная граница сегмента
    bucket_max - максимальная граница сегмента
    title - название сегмента

Таблица lifetime_segment содержит данные о выделяемых сегментах по количеству месяцев «жизни» клиента — лайфтайму:

    lt_gr_id - идентификатор сегмента, первичный ключ
    bucket_min - минимальная граница сегмента
    bucket_max - максимальная граница сегмента
    title - название сегмента

# Данное исследование разделим на несколько частей

## Импортируем необходимые библиотеки

In [1]:
import os
import pandas as pd
import numpy as np
import requests
from sqlalchemy import create_engine

## Пропишем путь к базе данных

In [2]:
path_to_db_local = r'C:\Users\Равиль\Проекты для портфолио\7. Проект NPS\telecomm_csi.db'
path_to_db_platform = '/telecomm_csi.db'
path_to_db = None

if os.path.exists(path_to_db_local):
    path_to_db = path_to_db_local
else:
    response = requests.get(path_to_db_platform)
    if response.status_code == 200:
        with open(path_to_db_local, 'wb') as f:
            f.write(response.content)
        path_to_db = path_to_db_local
    else:
        raise Exception('Файл с базой данных SQLite не найден!')

if path_to_db:
    engine = create_engine(f'sqlite:///{path_to_db}', echo=False)

## Выполним SQL запрос для выгрузки необходимых данных из таблиц базы

In [3]:
query = """
SELECT user_id,
       lt_day,
       CASE
           WHEN lt_day <= 365 THEN 'новый'
           WHEN lt_day > 365 THEN 'старый'
       END is_new,
       age,
       CASE
           WHEN gender_segment = 1 THEN 'female'
           WHEN gender_segment = 0 THEN 'male'
           ELSE 'unknown'
       END gender_segment,
       os_name,
       cpe_type_name,
       country,
       city,
       SUBSTRING(age_segment.title, 3) as age_segment,
       SUBSTRING(traffic_segment.title, 3) as traffic_segment,
       SUBSTRING(lifetime_segment.title, 3) as lifetime_segment,
       nps_score,
       CASE
           WHEN nps_score > 8 THEN 'promoters'
           WHEN nps_score > 6 THEN 'passives'
           ELSE 'detractors'
       END nps_group
FROM user
LEFT JOIN location ON location.location_id = user.location_id
LEFT JOIN age_segment ON age_segment.age_gr_id = user.age_gr_id
LEFT JOIN traffic_segment ON traffic_segment.tr_gr_id = user.tr_gr_id
LEFT JOIN lifetime_segment ON lifetime_segment.lt_gr_id = user.lt_gr_id
"""

## Cчитаем данные в датафрейм, сохраним в переменную data и проверим результат выполнения SQL запроса

In [4]:
data = pd.read_sql(query, engine)
data.head(9).T

,0,1,2,3,4,5,6,7,8
user_id,A001A2,A001WF,A003Q7,A004TB,A004XT,A005O0,A0061R,A009KS,A00AES
lt_day,2320,2344,467,4190,1163,5501,1236,313,3238
is_new,старый,старый,старый,старый,старый,старый,старый,новый,старый
age,45.0,53.0,57.0,44.0,24.0,42.0,45.0,35.0,36.0
gender_segment,female,male,male,female,male,female,male,male,female
os_name,ANDROID,ANDROID,ANDROID,IOS,ANDROID,ANDROID,ANDROID,ANDROID,ANDROID
cpe_type_name,SMARTPHONE,SMARTPHONE,SMARTPHONE,SMARTPHONE,SMARTPHONE,SMARTPHONE,SMARTPHONE,SMARTPHONE,SMARTPHONE
country,Россия,Россия,Россия,Россия,Россия,Россия,Россия,Россия,Россия
city,Уфа,Киров,Москва,РостовнаДону,Рязань,Омск,Уфа,Москва,СанктПетербург
age_segment,45-54,45-54,55-64,35-44,16-24,35-44,45-54,35-44,35-44


## Сохраним полученный файл в формате .csv для дальнейшей работы в Tableau

In [5]:
data.to_csv('telecomm_csi_tableau.csv', index=False)

Результат работы по созданию дашборда в Tableau и выводы отображены по <a href='https://public.tableau.com/app/profile/ravil.ganeev/viz/NPS_17314163075230/sheet18'>ссылке.</a>